# Localisation and combination of solvers

So far, all solvers in our tutorials have been active everywhere: Each
solver holds its data in every (unrefined) cell of the mesh, and it updates
these data in each and every time step. *Localisation* is the opposite: A
solver is active only in a part of the mesh, i.e. it neither stores nor updates
any data elsewhere. Localisation comes in two flavours:

- **Localisation in space.** A solver is active only within a subdomain. The
  classic example is a posteriori limiting, where a robust Finite Volume
  solver takes over from a higher-order solver only where shocks arise.
- **Localisation in resolution.** A solver is active only on a certain mesh
  level or mesh resolution. Peano's spacetree holds all resolution levels
  anyway, i.e. we can tell one solver to live on one particular level, while
  another solver lives on a finer level of the very same mesh.

We start with the motivation for the latter flavour, before we discuss the
mechanics of localisation along the lines of the former. As soon as solvers
are localised, they typically have to exchange data where their active
regions meet or overlap. This is discussed in
@ref tutorials_exahype2_45_coupling-sharing-material-parameters.

## Localisation in resolution and uncertainty quantification

Localisation in resolution might look like an exotic feature, but it is a key
ingredient of many modern uncertainty quantification (UQ) methods. In UQ, we do
not run one simulation, but many: We are interested in the statistics of some
output quantity $Q$, such as the expected wave height at a buoy, where the input
data (initial conditions, material parameters, bathymetry, ...) are uncertain.
A plain Monte Carlo estimate averages $Q$ over $N$ samples. Its statistical
error decays with $N^{-1/2}$ only, so we need many samples, and each sample is a
full PDE solve on a fine mesh. This quickly becomes prohibitively expensive.

Multilevel Monte Carlo (MLMC) (Giles 2008; Giles 2015) reduces this cost by
exploiting a hierarchy of meshes $\ell = 0, 1, \ldots, L$ from coarse to fine.
It rewrites the expectation on the finest mesh as a telescoping sum

\begin{equation*}
\mathbb{E}[Q_L] = \mathbb{E}[Q_0] + \sum_{\ell=1}^{L} \mathbb{E}[Q_\ell - Q_{\ell-1}]
\end{equation*}

and estimates each term with its own number of samples. Most samples are taken
on the cheap coarse meshes. The corrections $Q_\ell - Q_{\ell-1}$ have a small
variance and hence need only a few of the expensive fine-mesh samples.
Mishra, Schwab and Šukys (2012) have shown that this pays off for nonlinear
hyperbolic systems of conservation laws such as the Euler or the shallow water
equations, i.e. for exactly the type of problems that ExaHyPE is written for.
Multi-index Monte Carlo (Haji-Ali, Nobile and Tempone 2016) generalises the
idea to several hierarchies at once, such as the mesh size and the time step
size.

The same idea carries over to Bayesian inverse problems, where we infer
uncertain parameters from measurements through Markov chain Monte Carlo (MCMC):
Multilevel MCMC (Dodwell et al. 2015) and multilevel delayed acceptance
(Lykkegaard et al. 2023) use cheap coarse-mesh solves to steer or screen
proposals before the expensive fine-mesh solve is invoked. Seelinger et al.
(2021) have parallelised multilevel MCMC on supercomputers and identified the
source of a tsunami from buoy data, using ExaHyPE (Reinarz et al. 2020) as
forward solver. All of these approaches belong to the wider family of
multi-fidelity methods (Peherstorfer, Willcox and Gunzburger 2018), which
combine cheap low-fidelity models with expensive high-fidelity ones.

Where does localisation come into play? The crucial term in the telescoping
sum is the correction $Q_\ell - Q_{\ell-1}$: Both $Q_\ell$ and $Q_{\ell-1}$ have
to be computed for the very same sample, i.e. with the very same uncertain
input. Otherwise, the two do not correlate, and the variance of their
difference is not small. Typical UQ codes realise this through two independent
simulations, one per mesh, and they have to ensure that both see the same
input. With localisation in resolution, we can realise both simulations within
one mesh instead: One solver instance lives on level $\ell$ of the spacetree,
the other one on level $\ell-1$. Two adjacent levels of Peano's spacetree
differ by a factor of three in the mesh size, which yields a natural level
hierarchy. Both solvers share the mesh, the input data, the parallelisation
and the domain decomposition, and their difference can be computed on-the-fly,
cell by cell, rather than in a postprocessing step. The same reasoning holds
for the pairs of coarse and fine models within multilevel MCMC.

## Two Euler solvers on two resolutions

We now realise the idea behind the telescoping sum within one simulation: We
take the Euler equations from @ref tutorials_exahype2_10_euler and solve them
twice within the same mesh. One solver, EulerFV, runs on a fine mesh. A second
solver, CoarseEuler, uses exactly the same routines, but it lives on one coarse
level of the spacetree only. Both see the very same initial condition, i.e.
their outputs correspond to $Q_\ell$ and $Q_{\ell-1}$ for one sample.

### The fine solver

The fine solver is the solver from @ref tutorials_exahype2_10_euler. We only
ask for smaller volumes, i.e. a finer mesh:

In [ ]:
import sys, os

sys.path.insert(0, os.path.abspath("../../src"))

import peano4.api
import exahype2

In [ ]:
project = exahype2.api.Project(
    namespace    = ["tutorials", "exahype2", "localisation"],
    project_name = "Euler",
    directory    = "44_localisation",
    executable   = "euler",
)

patch_size  = 9
domain_size = [0.2, 0.2]

fine_solver = exahype2.api.Solver(
    "EulerFV",
    4,   # unknowns
)
fine_solver.add_pde_term(exahype2.api.Flux(stateless=False))
fine_solver.add_pde_term(exahype2.api.MaxEigenvalue(stateless=False))
fine_solver.add_pde_term(exahype2.api.InitPointWisely(stateless=False))

fine_solver.numerical_scheme  = exahype2.api.fv.RusanovWithExplicitEuler(fine_solver, patch_size)
fine_solver.grid_construction = exahype2.api.fv.SetupSimulationOverRegularGrid(fine_solver, volume_size=0.001)
fine_solver.plotter           = exahype2.api.fv.PlotAsVTKFile(
    fine_solver,
    first_plot_time_stamp = 0.0,
    time_in_between_plots = 0.005,
)

### A second solver with the same routines

The second solver needs a flux, an eigenvalue and an initial condition, too,
and they should be the very same ones. We could copy EulerFV's implementation
file, but then we would maintain two copies of the same physics. Instead, we
let CoarseEuler forward each call to EulerFV.

ExaHyPE creates one global instance per solver, which lives in the namespace
repositories of our project: repositories::instanceOfEulerFV is the fine solver
object. As we inject all PDE terms of CoarseEuler, ExaHyPE does not generate a
class CoarseEuler for us to fill. The injected code ends up in the
implementation file AbstractCoarseEuler.cpp, and so do the solver includes.
Therefore, we can include the solver repository there without creating a cyclic
include:

In [ ]:
coarse_solver = exahype2.api.Solver(
    "CoarseEuler",
    4,   # unknowns
)
coarse_solver.add_user_solver_includes('#include "repositories/SolverRepository.h"\n')

coarse_solver.add_pde_term(exahype2.api.Flux(
    implementation = "repositories::instanceOfEulerFV.flux(Q, x, normal, F);",
    stateless      = False,
))
coarse_solver.add_pde_term(exahype2.api.MaxEigenvalue(
    implementation = "repositories::instanceOfEulerFV.maxEigenvalue(Q, x, normal, lambdaMax);",
    stateless      = False,
))
coarse_solver.add_pde_term(exahype2.api.InitPointWisely(
    implementation = "repositories::instanceOfEulerFV.initPointWisely(x, Q);",
    stateless      = False,
))

### Running a solver on one level of the spacetree only

Each solver holds a grid selector. The selector decides on which vertices,
faces and cells all of the solver's action sets act, and where Peano stores
the solver's data. By default, a solver uses a FineGridSelector, i.e. it works
on the finest mesh level only.

CoarseEuler gets a FixedResolutionGridSelector instead. It selects all mesh
entities of one level of the spacetree, no matter whether the mesh is refined
further there or not. Level 0 is the root of the spacetree, i.e. the whole
domain, and each level splits its cells into $3 \times 3$ children. Level 2
hence consists of $9 \times 9$ cells, and with patches of $9 \times 9$ volumes,
CoarseEuler works on $81 \times 81$ volumes. The fine solver asked for volumes
of size 0.001, so ExaHyPE refines the mesh down to level 3. EulerFV hence works
on $243 \times 243$ volumes.

In [ ]:
coarse_solver.grid_selector     = peano4.api.toolbox.FixedResolutionGridSelector(2, domain_size)
coarse_solver.numerical_scheme  = exahype2.api.fv.RusanovWithExplicitEuler(coarse_solver, patch_size)
coarse_solver.grid_construction = exahype2.api.fv.SetupSimulationOverRegularGrid(coarse_solver, volume_size=0.005)
coarse_solver.plotter           = exahype2.api.fv.PlotAsVTKFile(
    coarse_solver,
    first_plot_time_stamp = 0.0,
    time_in_between_plots = 0.005,
)

Both solvers construct a regular grid. CoarseEuler's grid construction asks
for level 2 only, EulerFV's for level 3. The mesh is the union of the two, i.e.
it is refined down to level 3 everywhere, and each solver picks its level
through its selector.

The remaining steps are the same as in @ref tutorials_exahype2_10_euler. We
add both solvers to the project, and we write the implementation of EulerFV.

In [ ]:
project.add_solver(fine_solver)
project.add_solver(coarse_solver)

project.set_global_simulation_parameters(
    dimensions   = 2,
    offset       = [-0.1, -0.1],
    size         = domain_size,
    min_end_time = 0.2,
    max_end_time = 0.2,
    periodic_BC  = [True, True, True],
)

project.set_Peano4_installation(os.path.abspath("../../"), peano4.api.output.CompileMode.RELEASE)

peano4_project = project.generate_Peano4_project()
peano4_project.generate()

In [ ]:
!ls 44_localisation

There is a class EulerFV for us to implement, but no class CoarseEuler. We fill
EulerFV.cpp with the implementation from @ref tutorials_exahype2_10_euler, i.e.
the code which CoarseEuler forwards to:

In [ ]:
%%writefile 44_localisation/EulerFV.cpp
#include "EulerFV.h"


void tutorials::exahype2::localisation::EulerFV::flux(
  const double* __restrict__ Q, const tarch::la::Vector<Dimensions, double>& x, int normal, double* __restrict__ F
) {
  const double irho  = 1.0 / Q[0];
  constexpr double gamma = 1.4;
  const double p = (gamma - 1.0) * (Q[3] - 0.5 * irho * (Q[1] * Q[1] + Q[2] * Q[2]));

  switch (normal) {
    case 0:
      F[0] = Q[1];
      F[1] = irho * Q[1] * Q[1] + p;
      F[2] = irho * Q[1] * Q[2];
      F[3] = irho * Q[1] * (Q[3] + p);
      break;
    case 1:
      F[0] = Q[2];
      F[1] = irho * Q[2] * Q[1];
      F[2] = irho * Q[2] * Q[2] + p;
      F[3] = irho * Q[2] * (Q[3] + p);
      break;
  }
}


void tutorials::exahype2::localisation::EulerFV::maxEigenvalue(
  const double* __restrict__ Q, const tarch::la::Vector<Dimensions, double>& x, int normal, double* __restrict__ lambdaMax
) {
  const double irho  = 1.0 / Q[0];
  constexpr double gamma = 1.4;
  const double p = (gamma - 1.0) * (Q[3] - 0.5 * irho * (Q[1] * Q[1] + Q[2] * Q[2]));
  const double c = std::sqrt(gamma * p * irho);

  // velocity, i.e. momentum divided by density
  const double u = Q[1 + normal] * irho;

  lambdaMax[0] = std::max(std::abs(u + c), std::abs(u - c));
}


void tutorials::exahype2::localisation::EulerFV::initPointWisely(
  const tarch::la::Vector<Dimensions, double>& x, double* __restrict__ Q
) {
  // One period of a sine density profile over the periodic domain
  // [-0.1,0.1], constant velocity u and constant pressure p
  constexpr double gamma = 1.4;
  constexpr double p     = 1.0;
  constexpr double u     = 1.0;
  const double     rho   = 1.0 + 0.2 * std::sin(2.0 * M_PI * (x[0] + 0.1) / 0.2);
  Q[0] = rho;
  Q[1] = rho * u;
  Q[2] = 0.0;
  Q[3] = p / (gamma - 1) + 0.5 * rho * u * u;
}

In [ ]:
%%writefile 44_localisation/exahype2.log-filter
debug     tarch       -1      black     always-on
debug     peano4      -1      black     always-on
debug     exahype2    -1      black     always-on
debug     toolbox     -1      black     always-on
debug     tutorials   -1      black     always-on

trace     tarch       -1      black     always-on
trace     peano4      -1      black     always-on
trace     exahype2    -1      black     always-on
trace     toolbox     -1      black     always-on
trace     tutorials   -1      black     always-on

info      tarch       -1      white     always-on
info      peano4      -1      white     always-on
info      exahype2    -1      white     always-on
info      toolbox     -1      white     always-on
info      tutorials   -1      white     always-on

In [ ]:
!cd 44_localisation && make -j > make.log 2>&1 && ./euler > run.log 2>&1 && echo done

### Plotting

Each solver has its own plotter, so we get two sets of patch files:
solution-EulerFV and solution-CoarseEuler. Each solver decides on its own when
to plot, i.e. it plots as soon as its own time stamp has passed the next plot
time. The snapshots of the two solvers therefore carry slightly different time
stamps:

In [ ]:
import tarch.plotter.api.notebook

for name in ["EulerFV", "CoarseEuler"]:
    snapshots = tarch.plotter.api.notebook.read_snapshots(f"44_localisation/solution-{name}.peano-patch-file")
    print(f"{name:12}", len(snapshots), "snapshots, last ones at t =", [round(t, 4) for t, _ in snapshots[-3:]])

You can open both files in the notebook viewer, or convert both into VTK and
load both pvd files into Paraview. CoarseEuler's patches are three times as big
as EulerFV's patches, and they cover the same domain. Show one at a time, or
shrink or shift one of them to see both.

In [ ]:
tarch.plotter.api.notebook.show_solution("44_localisation/solution-EulerFV.peano-patch-file")

In [ ]:
tarch.plotter.api.notebook.show_solution("44_localisation/solution-CoarseEuler.peano-patch-file")

@note One solver runs ahead in time. Each solver holds its own time stamp and
picks its own time step size, which depends on its maximum eigenvalue. The
coarse solver smears out the solution more and hence sees slightly smaller
eigenvalues. It therefore runs slightly ahead of the fine one. The simulation
terminates as soon as the first solver has reached the end time, so EulerFV
stops a little before $t=0.2$. We leave this as it is for the time being: The
two solvers do not exchange any data so far, so they do not have to agree on
their time stamps.

### Comparing the two solvers

As the snapshots do not match exactly in time, we compare the two solvers in two
ways that cope with this.

First, we probe both solvers at the same point. Each probe yields a time series
with its own time axis, so we can plot the two series over each other without
pairing up snapshots:

In [ ]:
import csv
import subprocess
import matplotlib.pyplot as plt

probe_environment = os.environ.copy()
probe_environment["PYTHONPATH"] = os.path.abspath("../../src")


def probe(name, point):
    subprocess.run(
        [sys.executable, "-m", "tarch.plotter.api.probe", "--point", *[str(c) for c in point],
         "-o", f"probe-{name}.csv", f"solution-{name}.peano-patch-file"],
        cwd="44_localisation", env=probe_environment, check=True, stdout=subprocess.DEVNULL,
    )
    with open(f"44_localisation/probe-{name}.csv") as csv_file:
        rows = list(csv.reader(csv_file))
    return [float(row[0]) for row in rows[1:]], [float(row[1]) for row in rows[1:]]


fig, ax = plt.subplots(figsize=(7, 4))
for name in ["EulerFV", "CoarseEuler"]:
    t, rho = probe(name, (0.0, 0.0))
    ax.plot(t, rho, label=name)
ax.set_xlabel("t")
ax.set_ylabel("density at x=(0,0)")
ax.legend()
plt.show()

![Density at the centre of the domain over time](illustrations/44_localisation-probe.png)

The coarse solver's wave has a smaller amplitude. Furthermore, EulerFV's curve
ends a little earlier: the run stops once CoarseEuler has reached $t=0.2$.

Second, we compare whole snapshots. We pair each snapshot of one solver with
the snapshot of the other solver which is closest in time, and plot the two with
the same colour range:

In [ ]:
def closest_snapshot(file_name, time_stamp):
    snapshots = tarch.plotter.api.notebook.read_snapshots(file_name)
    return min(range(len(snapshots)), key=lambda i: abs(snapshots[i][0] - time_stamp))


fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, name in zip(axes, ["EulerFV", "CoarseEuler"]):
    file_name = f"44_localisation/solution-{name}.peano-patch-file"
    tarch.plotter.api.notebook.plot_snapshot(
        file_name, closest_snapshot(file_name, 0.1), 0, clim=(0.8, 1.2), ax=ax, label=f"{name}: density"
    )
plt.show()

![Density of the fine and the coarse solver at t=0.1](illustrations/44_localisation-comparison.png)

Both solvers advect the same density wave. The coarse solver however smears it
out more: Rusanov's numerical diffusion scales with the size of the
volumes. At $t=0.2$, the coarse density ranges from around 0.88 to 1.12, whereas
the fine solver still yields around 0.83 to 1.17. The initial condition
oscillates between 0.8 and 1.2.

The pictures are created by tutorials/exahype2/illustrations/44_localisation.py.

### Outlook: time step sizes per solver (not implemented yet)

**This subsection describes ideas only. None of them is implemented yet, i.e.
the code above behaves as described in the note, not as sketched here.**

Today, each solver computes its time step size from the CFL condition with its
own maximum eigenvalue, but with the smallest volume of the whole mesh: The
action set ComputeGlobalCFLCondition takes the mesh size from Peano's
grid statistics, which know nothing about solvers or grid selectors. CoarseEuler
therefore marches with time steps that fit EulerFV's volumes, although its own
volumes are three times as big.

The idea is to let each solver use the size of its own volumes instead,
reduced the same way as the maximum eigenvalue already is:

- The action set holds a further static attribute, the minimum volume size,
  which prepareTraversal() resets.
- touchCellFirstTime() folds the volume size of each cell which the solver's
  grid selector picks into this attribute, under the same lock as the
  eigenvalue. Only the solver's own cells contribute.
- unprepareTraversal() reduces the minimum across all ranks, and uses it in
  the CFL condition instead of the mesh size from the grid statistics.
- If a solver has no cells at all in a mesh sweep, it keeps its previous time
  step size.

With this change, CoarseEuler would take time steps roughly three times as big
as EulerFV's, and it would reach the end time after roughly a third of the time
steps. As the simulation terminates as soon as the first solver reaches the end
time, EulerFV would stop at around $t \approx 0.07$. A second idea addresses
this: The main loop could run until every solver has reached the end time. Each
solver already shortens its last time step such that it does not overshoot the
end time, i.e. a solver which is done would simply idle with time step size 0
until the others have caught up. Alternatively, the solvers could agree on one
common time step size. Which option is the right one depends on whether the
solvers exchange data, which is what the text part of this tutorial discusses.

## Localisation

ExaHyPE couples volumetrically. That does not mean that all solvers have to be
solved everywhere. Indeed, the idea to offer support for multiple solvers arose
originally from the a posteriori limiting of Dumbser et al where a Finite Volume
solver complements ADER-DG solutions where shocks arise.

Most projects that follow these rationale realise the same pattern:
We define two different solvers, and we couple them
with each other. The coupling (and solve) however are local operations:

-   The low-order (typically Finite Volume - we therefore always refer
    to Finite Volumes or FV from hereon) solver is our fallback strategy and we therefore
    neither solve with Finite Volumes everywhere in the computational
    domain nor should do we store the corresponding data.

-   The higher-order scheme is used everywhere where the FV is not
    applied or labelled as valid.

-   The FV and the higher-order solver are not coupled everywhere but only in
    boundary regions.

We will effectively end up with a domain decomposition where parts of
the computational domain are handled by the higher-order solver and the other
parts are handled with FV.
To realise this, we need a marker or oracle, i.e. a magic function
per cell which tells us, similar to a multi-phase flow, which
solver rules in this particular cell.
The
marker will guide us which solver is "valid" in which cell, where one solver
overrules the other one, and where we do not have to store one solver's data
or the other.

The image below illustrates the core idea behind typically coupling approaches
in ExaHyPE. The sizes of the overlap might differ, but the principle is always
the same. The illustration shows the minimal overlap that we can typically
use. Both the upper and the lower mesh illustrate the same mesh. I use colours
to show which solver exists in which mesh cell.

![Overlap of two localised solvers](illustrations/44_localisation.png)

- Solver A is running within the filled cells visualised on the the top. In
  empty cells, A neither solves anything nor does it hold data.
- Solver B is running within the filled cells visualised on the bottom. In
  empty cells, B neither solves anything nor does it hold data.
- Consequently, dark red cells are exclusively solved by A, whereas dark
  green cells are exclusively B.
- In orange cells, A solves its PDE. Afterwards, A's updated solution is
  projected onto B's solution. That is, the blue cells hold B's data, but
  they are not updated by B but instead hold projections of A's data. They
  kind of serve as boundary layer for B.
- In the yellow cells, A does not solve its PDE but holds A's data. After
  each step, the cell's solution by B (light green) is written into A's data.
  The yellow cells serve as Dirichlet boundary data for A's solve.

Starting from this description, sophisticated coupling equations can be
realised, where you exploit the fact that we have a volumetric overlap.

### Localisation guided by a marker

Skip this section if you prefer a hard-coded marking of active regions.

### Avoid that solver data is held everywhere

Once you know where a solver is valid, it makes sense also to only store
the solver's data in the valid region. Otherwise, a lot of memory is "wasted"
which furthermore will be moved/accessed in each and every grid sweep, as the
mesh management of Peano so far does not know that ExaHyPE might not need the
data. ExaHyPE has to let Peano know.

The technical principles behind such a localisation @ref tutorials_peano4_mask_out_data_usage "are discussed in a Peano tutorial of its own".
ExaHyPE builds up pretty deep inheritance hierarchy
where every subclass specialises and augments the basic numerical scheme, and
also implements additional guards when data is stored or not.
However, most of these subclasses rely on only six routines to control
the data flow:

- _provide_cell_data_to_compute_kernels_default_guard(),
- _load_cell_data_default_guard(),
- _store_cell_data_default_guard(),
- _provide_face_data_to_compute_kernels_default_guard(),
- _load_face_data_default_guard(),
- _store_face_data_default_guard(),

The implementation then uses these predicates to puzzle the actual storage
scheme together via calls similar to

```python
        self._patch.generator.load_store_compute_flag = "::peano4::grid::constructLoadStoreComputeFlag({},{},{})".format(
            self._provide_cell_data_to_compute_kernels_default_guard(),
            self._load_cell_data_default_guard(),
            self._store_cell_data_default_guard(),
        )
```

By default, most solvers only store data for unrefined
cells. They mask coarser levels with in the tree out.
Subclasses such as enclave tasking variations augment the guards
further such that not all data are stored all the time. Nothing stops you
from adding further constraints and to use a solver only in a certain
subdomain for example.

## References

- M. B. Giles: [Multilevel Monte Carlo path simulation](https://doi.org/10.1287/opre.1070.0496).
  Operations Research, 56(3), pp. 607-617 (2008)
- M. B. Giles: [Multilevel Monte Carlo methods](https://doi.org/10.1017/S096249291500001X).
  Acta Numerica, 24, pp. 259-328 (2015)
- S. Mishra, Ch. Schwab, J. Šukys: [Multi-level Monte Carlo finite volume methods for nonlinear systems of conservation laws in multi-dimensions](https://doi.org/10.1016/j.jcp.2012.01.011).
  Journal of Computational Physics, 231(8), pp. 3365-3388 (2012)
- A.-L. Haji-Ali, F. Nobile, R. Tempone: [Multi-index Monte Carlo: when sparsity meets sampling](https://doi.org/10.1007/s00211-015-0734-5).
  Numerische Mathematik, 132, pp. 767-806 (2016)
- T. J. Dodwell, C. Ketelsen, R. Scheichl, A. L. Teckentrup: [A hierarchical multilevel Markov chain Monte Carlo algorithm with applications to uncertainty quantification in subsurface flow](https://doi.org/10.1137/130915005).
  SIAM/ASA Journal on Uncertainty Quantification, 3(1), pp. 1075-1108 (2015)
- M. B. Lykkegaard, T. J. Dodwell, C. Fox, G. Mingas, R. Scheichl: [Multilevel delayed acceptance MCMC](https://doi.org/10.1137/22M1476770).
  SIAM/ASA Journal on Uncertainty Quantification, 11, pp. 1-30 (2023)
- L. Seelinger, A. Reinarz, L. Rannabauer, M. Bader, P. Bastian, R. Scheichl: [High performance uncertainty quantification with parallelized multilevel Markov chain Monte Carlo](https://doi.org/10.1145/3458817.3476150).
  Proceedings of the International Conference for High Performance Computing, Networking, Storage and Analysis (SC21), pp. 1-15 (2021)
- A. Reinarz, D. E. Charrier, M. Bader et al.: [ExaHyPE: An engine for parallel dynamically adaptive simulations of wave problems](https://doi.org/10.1016/j.cpc.2020.107251).
  Computer Physics Communications, 254, 107251 (2020)
- B. Peherstorfer, K. Willcox, M. Gunzburger: [Survey of multifidelity methods in uncertainty propagation, inference, and optimization](https://doi.org/10.1137/16M1082469).
  SIAM Review, 60(3), pp. 550-591 (2018)